# Milestone 2 — Exploratory Data Analysis (EDA): UP District-Level Crop Yield Dataset

This notebook performs comprehensive Exploratory Data Analysis on the **Uttar Pradesh District-Level Rice & Wheat Historical Yield Dataset (`data/raw/yield/up_district_yield_apy_1997_2023.csv`)** covering agricultural years **1997–1998 to 2023–2024 across 75 administrative districts**.

## Objectives (Instructor Guidelines Sections 2, 3, 4 & 5)
1. **Dataset Profile & Missing Value Audit:** Analyze dataset completeness, administrative bifurcations (Amethi, Sambhal, Hapur, Shamli), and bureaucratic zero-production reporting anomalies.
2. **Target Variable Analysis (`Yield_Kg_Ha`):** Quantify central tendencies, dispersion, and distributions for Kharif Rice and Rabi Wheat.
3. **Agro-Climatic Spatial Disparities:** Compare Western UP, Central UP, Eastern UP, and Bundelkhand yield & irrigation performance.
4. **Longitudinal Trends & Agronomic Intensification:** Evaluate Green Revolution NPK fertilizer and tubewell expansion over 27 years.
5. **Meteorological Stress Correlation Analysis:** Link IMD seasonal rainfall, extreme flood days (`Rain_Days_Extreme`), and March heatwaves (`Heatwave_Days`) to crop yield.

In [1]:
import os
import csv
import math

DATA_PATH = os.path.join("..", "data", "raw", "yield", "up_district_yield_apy_1997_2023.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("data", "raw", "yield", "up_district_yield_apy_1997_2023.csv")

records = []
with open(DATA_PATH, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"Loaded {len(records)} district-year-season records from {DATA_PATH}")

Loaded 3886 district-year-season records from ..\data\raw\yield\up_district_yield_apy_1997_2023.csv


## 1. Missingness & District Bifurcation Audit
Administrative districts in Uttar Pradesh expanded over the 1997–2023 timeline. Districts carved out after 1997 lack independent records prior to their formation dates.

In [2]:
expected_total = 75 * 27 * 2
bifurcation_missing = expected_total - len(records)
zero_prod = [r for r in records if float(r["Production_Total"]) == 0.0]

print("=== DATASET COMPLETENESS AUDIT ===")
print(f"Total Expected Records (75 districts x 27 years x 2 seasons): {expected_total}")
print(f"Observed Records: {len(records)}")
print(f"Records Absent due to Historical District Bifurcations: {bifurcation_missing} ({bifurcation_missing/expected_total*100:.1f}%)")
print(f"Sporadic Zero-Production Reporting Anomalies: {len(zero_prod)} ({len(zero_prod)/len(records)*100:.2f}%)")

=== DATASET COMPLETENESS AUDIT ===
Total Expected Records (75 districts x 27 years x 2 seasons): 4050
Observed Records: 3886
Records Absent due to Historical District Bifurcations: 164 (4.0%)
Sporadic Zero-Production Reporting Anomalies: 29 (0.75%)


## 2. Target Variable Summary Statistics (`Yield_Kg_Ha`)
Excluding the 0.75% sporadic reporting anomalies, we inspect summary statistics across both crops.

In [3]:
def stats(vals):
    mean = sum(vals) / len(vals)
    std = math.sqrt(sum((x - mean)**2 for x in vals) / len(vals))
    s = sorted(vals)
    n = len(s)
    return mean, std, s[n//2], s[n//4], s[(3*n)//4]

for crop in ["Rice", "Wheat"]:
    y = [float(r["Yield_Kg_Ha"]) for r in records if r["Crop"] == crop and float(r["Yield_Kg_Ha"]) > 0]
    mean, std, med, q1, q3 = stats(y)
    print(f"{crop.upper()} YIELD SUMMARY (kg/ha):")
    print(f"  Count: {len(y)} | Mean: {mean:.1f} | Std: {std:.1f} | Median: {med:.1f}")
    print(f"  IQR: {q3-q1:.1f} (Q1: {q1:.1f}, Q3: {q3:.1f}) | Range: {min(y):.1f} - {max(y):.1f}\n")

RICE YIELD SUMMARY (kg/ha):
  Count: 1933 | Mean: 2459.9 | Std: 466.5 | Median: 2447.4
  IQR: 628.2 (Q1: 2144.7, Q3: 2772.9) | Range: 1134.1 - 3917.4

WHEAT YIELD SUMMARY (kg/ha):
  Count: 1924 | Mean: 2786.2 | Std: 750.7 | Median: 2751.0
  IQR: 980.5 (Q1: 2302.8, Q3: 3283.3) | Range: 920.3 - 4750.0



## 3. Agro-Climatic Zone Disparities
Uttar Pradesh exhibits sharp regional contrasts between canal/tubewell-rich Western UP and drought-prone Bundelkhand.

In [4]:
zones = ["Western UP", "Central UP", "Eastern UP", "Bundelkhand"]
print(f"{'Zone':<15} | {'Rice Yield':<11} | {'Wheat Yield':<11} | {'Net Irrig %':<11} | {'Tubewell %':<11}")
print("-" * 68)
for zone in zones:
    ry = [float(r["Yield_Kg_Ha"]) for r in records if r["Agro_Climatic_Zone"] == zone and r["Crop"] == "Rice" and float(r["Yield_Kg_Ha"]) > 0]
    wy = [float(r["Yield_Kg_Ha"]) for r in records if r["Agro_Climatic_Zone"] == zone and r["Crop"] == "Wheat" and float(r["Yield_Kg_Ha"]) > 0]
    ir = [float(r["Net_Irrigated_Pct"]) for r in records if r["Agro_Climatic_Zone"] == zone and r["Crop"] == "Wheat"]
    tb = [float(r["Tubewell_Irrig_Pct"]) for r in records if r["Agro_Climatic_Zone"] == zone and r["Crop"] == "Wheat"]
    print(f"{zone:<15} | {stats(ry)[0]:<11.1f} | {stats(wy)[0]:<11.1f} | {stats(ir)[0]:<11.1f} | {stats(tb)[0]:<11.1f}")

Zone            | Rice Yield  | Wheat Yield | Net Irrig % | Tubewell % 
--------------------------------------------------------------------
Western UP      | 2893.8      | 3551.9      | 94.0        | 86.0       
Central UP      | 2403.9      | 2747.4      | 82.7        | 77.8       
Eastern UP      | 2292.3      | 2444.6      | 74.5        | 72.9       
Bundelkhand     | 1719.7      | 1490.8      | 50.6        | 41.8       


## 4. Covariate Correlation Matrix
Evaluates Pearson correlation between yield and meteorological/agronomic drivers.

In [5]:
def corr(xs, ys):
    mx = sum(xs) / len(xs)
    my = sum(ys) / len(ys)
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = math.sqrt(sum((x - mx)**2 for x in xs) * sum((y - my)**2 for y in ys))
    return num / den if den != 0 else 0.0

for crop in ["Rice", "Wheat"]:
    crecs = [r for r in records if r["Crop"] == crop and float(r["Yield_Kg_Ha"]) > 0]
    y = [float(r["Yield_Kg_Ha"]) for r in crecs]
    rain = [float(r["Precip_Seasonal_mm"]) for r in crecs]
    irrig = [float(r["Net_Irrigated_Pct"]) for r in crecs]
    extreme = [float(r["Rain_Days_Extreme"]) for r in crecs]
    heat = [float(r["Heatwave_Days"]) for r in crecs]
    n_fert = [float(r["Fertilizer_N_Tonnes"]) / max(1.0, float(r["Area_Sown"])) * 1000 for r in crecs]
    print(f"{crop.upper()} CORRELATIONS WITH YIELD:")
    print(f"  Net Irrigated %:        {corr(y, irrig):+.3f}")
    print(f"  Fertilizer N Intensity: {corr(y, n_fert):+.3f}")
    print(f"  Seasonal Precip (mm):   {corr(y, rain):+.3f}")
    print(f"  Extreme Rain Days:      {corr(y, extreme):+.3f}")
    if crop == "Wheat":
        print(f"  March Heatwave Days:    {corr(y, heat):+.3f}")
    print()

RICE CORRELATIONS WITH YIELD:
  Net Irrigated %:        +0.813
  Fertilizer N Intensity: +0.803
  Seasonal Precip (mm):   +0.034
  Extreme Rain Days:      -0.223

WHEAT CORRELATIONS WITH YIELD:
  Net Irrigated %:        +0.899
  Fertilizer N Intensity: +0.864
  Seasonal Precip (mm):   +0.356
  Extreme Rain Days:      +0.000
  March Heatwave Days:    -0.241

